In [1]:
import torch
from torch import nn

In [2]:
from torch.utils.tensorboard import SummaryWriter

In [3]:
import os
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

NUM_WORKERS = os.cpu_count()
#NUM_WORKERS = 0 # you can make this 0 if you get any error on mac silicon

def create_data(
    train_dir: str,
    test_dir: str,
    transform: transforms.Compose,
    batch_size: int,
    num_workers: int=NUM_WORKERS
):
  # Use ImageFolder to create dataset(s)
  train_data = datasets.ImageFolder(train_dir, transform=transform)
  test_data = datasets.ImageFolder(test_dir, transform=transform)

  # Get class names
  class_names = train_data.classes

  # Turn images into data loaders
  train_dataloader = DataLoader(
      train_data,
      batch_size=batch_size,
      shuffle=True,
      num_workers=num_workers,
      pin_memory=True,
  )
  test_dataloader = DataLoader(
      test_data,
      batch_size=batch_size,
      shuffle=False, # don't need to shuffle test data
      num_workers=num_workers,
      pin_memory=True,
  )

  return train_dataloader, test_dataloader, class_names

In [4]:
class DesertClassifier(nn.Module):
    def __init__(self, input_shape: int, hidden_units: int, output_shape: int) -> None:
        super().__init__()
        self.conv_block_1 = nn.Sequential(
            nn.Conv2d(in_channels=input_shape,
                      out_channels=hidden_units,
                      kernel_size=3,
                      stride=1,
                      padding=1),
            nn.ReLU(),
            nn.Conv2d(in_channels=hidden_units,
                      out_channels=hidden_units,
                      kernel_size=3,
                      stride=1,
                      padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2,
                         stride=2)  # default stride value is same as kernel_size
        )
        self.conv_block_2 = nn.Sequential(
            nn.Conv2d(hidden_units, hidden_units, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(hidden_units, hidden_units, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features=hidden_units * 16 * 16,
                      out_features=output_shape)
        )

    def forward(self, x: torch.Tensor):
        return self.classifier(self.conv_block_2(self.conv_block_1(x)))

In [5]:
import torch
from torch import nn

def calculate_accuracy(y_pred, y_true):
    correct = torch.eq(y_pred, y_true).sum().item()
    acc = (correct / len(y_true)) * 100
    return acc

def train_step(model: torch.nn.Module,
               dataloader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module,
               optimizer: torch.optim.Optimizer,
               device,
               accuracy_function):
    # Training Mode
    model.train()
    train_loss, train_acc = 0, 0

    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)
        y_pred = model(X)

        loss = loss_fn(y_pred, y)
        train_loss += loss.item()
        train_acc += accuracy_function(y_pred.argmax(dim = 1), y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    train_loss /= len(dataloader)
    train_acc /= len(dataloader)

    return train_loss, train_acc

def test_step(model: torch.nn.Module,
              dataloader: torch.utils.data.DataLoader,
              loss_fn: torch.nn.Module,
              device,
              accuracy_function):
    # Testing Step
    model.eval()
    test_loss, test_acc = 0, 0

    with torch.inference_mode():
        for batch, (X, y) in enumerate(dataloader):
            X, y = X.to(device), y.to(device)
            test_pred = model(X)

            loss = loss_fn(test_pred, y)
            test_loss += loss.item()
            test_acc += accuracy_function(test_pred.argmax(dim = 1), y)

    test_loss /= len(dataloader)
    test_acc /= len(dataloader)

    return test_loss, test_acc

def train(model: torch.nn.Module,
          train_dataloader: torch.utils.data.DataLoader,
          test_dataloader: torch.utils.data.DataLoader,
          optimizer: torch.optim.Optimizer,
          loss_fn: torch.nn.Module,
          epochs: int,
          device,
          accuracy_function,
          experiment_name = "ex1"):

    # 1-Sonuçları tutmak için boş bir sözlük oluşturdum
    results = {
        "train_loss" : [],
        "train_acc" : [],
        "test_loss" : [],
        "test_acc" : []
    }

    # 2-Eğitim döngüsünü kurdum, train ve test fonksiyonlarını çağırarak eğitimi yaptım.
    for epoch in range(epochs):
        train_loss, train_acc = train_step(model=model,
                                           dataloader=train_dataloader,
                                           loss_fn=loss_fn,
                                           optimizer=optimizer,
                                           device=device,
                                           accuracy_function=accuracy_function)
        test_loss, test_acc = test_step(model=model,
                                        dataloader=test_dataloader,
                                        loss_fn=loss_fn,
                                        device=device,
                                        accuracy_function=accuracy_function)

        # 3-Çıkan sonuçları yazdırdım
        print(f"Epoch: {epoch+1} | Train_loss: {train_loss:.4f} | Train Accuracy: {train_acc:.4f} | Test Loss: {test_loss:.4f} | Test Accuracy: {test_acc:.4f}")

        results["train_loss"].append(train_loss.item() if isinstance(train_loss, torch.Tensor) else train_loss)
        results["train_acc"].append(train_acc.item() if isinstance(train_acc, torch.Tensor) else train_acc)
        results["test_loss"].append(test_loss.item() if isinstance(test_loss, torch.Tensor) else test_loss)
        results["test_acc"].append(test_acc.item() if isinstance(test_acc, torch.Tensor) else test_acc)


        # Tensorboard Kısmı
        writer = SummaryWriter(log_dir=f"runs/{experiment_name}")
        writer.add_scalars(
            main_tag="Loss",
            tag_scalar_dict={
                "train_loss" : train_loss,
                "test_loss" : test_loss
            },
            global_step=epoch
        )
        writer.add_scalars(
            main_tag="Accuracy",
            tag_scalar_dict={
                "train_acc" : train_acc,
                "test_acc" : test_acc
            },
            global_step=epoch
        )
        writer.add_graph(
            model=model,
            input_to_model=torch.rand(32, 3, 64, 64)
        )

        writ.close()

    return results

In [6]:

    train_dir = "data/desert101/train"
    test_dir = "data/desert101/test"

    BATCH_SIZE = 32
    LEARNING_RATE = 0.001
    NUM_EPOCHS = 10

    data_transform = transforms.Compose([
        transforms.Resize(size=(64,64)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(degrees=15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5483, 0.4638, 0.3865],
                             std=[0.2173, 0.2279, 0.2263])
    ])

    train_dataloader, test_dataloader, class_names = create_data(
        train_dir=train_dir,
        test_dir=test_dir,
        batch_size=BATCH_SIZE,
        transform=data_transform
    )
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model_1 = DesertClassifier(
        input_shape=3,
        hidden_units=10,
        output_shape=len(class_names)
    )
    model_1 = model_1.to(device)

    model_2 = DesertClassifier(
        input_shape=3,
        hidden_units=32,
        output_shape=len(class_names)
    )
    model_2 = model_2.to(device)


In [7]:
    loss_fn = torch.nn.CrossEntropyLoss()
    optimizer_1 = torch.optim.Adam(
        params=model_1.parameters(),
        lr=LEARNING_RATE
    )
    optimizer_2 = torch.optim.Adam(
        params=model_2.parameters(),
        lr=LEARNING_RATE
    )

In [9]:
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    train(
        model=model_1,
        train_dataloader=train_dataloader,
        test_dataloader=test_dataloader,
        optimizer=optimizer_1,
        loss_fn=loss_fn,
        epochs=NUM_EPOCHS,
        device=device,
        accuracy_function=calculate_accuracy
    )

Epoch: 1 | Train_loss: 1.3842 | Train Accuracy: 28.5268 | Test Loss: 1.3807 | Test Accuracy: 32.2917


FailedPreconditionError: runs/ex1 is not a directory

In [ ]:
train(
    model=model_2,
    train_dataloader=train_dataloader,
    test_dataloader=test_dataloader,
    loss_fn=loss_fn,
    optimizer=optimizer_2,
    epochs=NUM_EPOCHS,
    experiment_name="hidden_units_32"
)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs # run session with the "runs/" directory (runs directory is created automatically)